# Project Template: Regularization Comparison Study

- Module band: 01-04
- Estimated runtime: 10-15 minutes for the starter notebook
- Estimated project time: 5-7 hours
- Prerequisites: least squares, ridge, lasso, validation-based model selection
- Expected outputs: validation comparison table, test metrics, coefficient comparison, written interpretation


## Learning goals

- Compare OLS, ridge, and lasso on the same noisy regression problem.
- Observe how regularization changes coefficient behavior and generalization.
- Connect empirical results back to bias-variance reasoning.
- Write a short recommendation for when each method is appropriate.


In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline

SEED = 7
rng = np.random.default_rng(SEED)

x = np.linspace(-3.0, 3.0, 180)
x = x + rng.normal(0.0, 0.08, size=x.shape[0])
y = 1.5 + 2.0 * x - 0.9 * x**2 + 0.35 * x**3 + rng.normal(0.0, 1.4, size=x.shape[0])
X = x[:, None]

X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.25, random_state=SEED)
X_train.shape, X_val.shape, X_test.shape


## Step 1: State your expectations

Add a short response here:

- Why might an unregularized polynomial model overfit?
- What do you expect ridge to change?
- What do you expect lasso to change?


In [ ]:
def make_pipeline(model):
    return Pipeline(
        steps=[
            ("poly", PolynomialFeatures(degree=8, include_bias=False)),
            ("scale", StandardScaler()),
            ("model", model),
        ]
    )


candidate_models = {
    "ols": make_pipeline(LinearRegression()),
    "ridge_0.1": make_pipeline(Ridge(alpha=0.1)),
    "ridge_2.0": make_pipeline(Ridge(alpha=2.0)),
    "lasso_0.01": make_pipeline(Lasso(alpha=0.01, max_iter=10000)),
    "lasso_0.05": make_pipeline(Lasso(alpha=0.05, max_iter=10000)),
}

validation_results = []
for name, model in candidate_models.items():
    model.fit(X_train, y_train)
    val_pred = model.predict(X_val)
    validation_results.append(
        {
            "model": name,
            "val_rmse": mean_squared_error(y_val, val_pred) ** 0.5,
        }
    )
validation_results

# TODO:
# 1. Add or adjust regularization strengths.
# 2. Choose one final ridge model and one final lasso model.


In [ ]:
final_models = {
    name: model.fit(X_train_full, y_train_full)
    for name, model in {
        "ols": candidate_models["ols"],
        "ridge": candidate_models["ridge_2.0"],
        "lasso": candidate_models["lasso_0.05"],
    }.items()
}

test_results = {}
for name, model in final_models.items():
    pred = model.predict(X_test)
    test_results[name] = {"rmse": mean_squared_error(y_test, pred) ** 0.5}
test_results


In [ ]:
grid = np.linspace(X.min(), X.max(), 300)[:, None]
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].scatter(X_train[:, 0], y_train, s=18, alpha=0.6, label="train")
axes[0].scatter(X_val[:, 0], y_val, s=18, alpha=0.6, label="val")
for name, model in final_models.items():
    axes[0].plot(grid[:, 0], model.predict(grid), label=name)
axes[0].set_title("Fitted curves")
axes[0].legend()

coefficient_counts = {}
for name, model in final_models.items():
    coef = model.named_steps["model"].coef_
    coefficient_counts[name] = {
        "l1_norm": float(np.abs(coef).sum()),
        "near_zero": int((np.abs(coef) < 1e-3).sum()),
    }
axes[1].bar(coefficient_counts.keys(), [item["l1_norm"] for item in coefficient_counts.values()])
axes[1].set_title("Coefficient L1 norm by model")
plt.tight_layout()

coefficient_counts


## Final analysis prompts

Answer these in complete sentences:

1. Which model generalized best, and what evidence supports that claim?
2. How did ridge and lasso differ in coefficient behavior?
3. What signs of overfitting do you see in the unregularized model?
4. When would you prefer ridge, and when would you prefer lasso?
